# Module 4 — Structured Mazes, Spikes & Data Extraction

Modules 1 to 3 all used an open arena. Many experiments use a maze or some type of track environment to create a task for the animal, e.g. running down a hallway then learning to go right to receive a reward (like a cheerio). In this module we will create a structured layout, and observe how neurons behave near barriers and how to package spike data into a format we can train in a model.

> **Structured Environments**: A maze constrains the animal to a small number of stereotyped routes. Hippocampal firing on a maze can depend on where the animal came from (Wood et al., 2000) or where it is going next (Frank et al., 2000), not just on where it is right now. That route dependence is one reason the same cell population can look different on a maze than it does in an open field (McNaughton et al., 1983). RatInABox builds mazes by adding wall segments to an `Environment` (George et al., 2024).

## Outcomes from this Module

- Build a maze with `env.add_wall([[x0, y0], [x1, y1]])` and read the segments back off `env.walls`
- Know that a fresh `Environment` already carries **four** boundary walls, so `len(env.walls)` is four more than you added
- Draw the layout with `env.plot_environment()` and overlay a run with `Ag.plot_trajectory()`
- **Verify** containment numerically instead of trusting the picture
- Choose a `wall_geometry` for `PlaceCells` from `"geodesic"`, `"euclidean"` and `"line_of_sight"`
- Recognise the **silent** `geodesic` fallback on any maze with more than one added wall
- Pull `firingrate` and `spikes` out of `layer.get_history_arrays()`
- Know that `spikes` is a **boolean** array, at most one spike per cell per `dt`
- Measure the saturation that follows from that at a high `max_fr`
- Place a goal with `env.add_object([x, y])` and read it back from `env.objects["objects"]`
- Measure distance to that goal with `env.get_distances_between___accounting_for_environment()`
- Handle the hard `AssertionError` that `geodesic` raises there, which is **not** a fallback
- Write a reusable `extract_dataset(Ag, layers)` returning aligned `(T, ...)` arrays
- Chunk a `(T, F)` matrix into overlapping windows and split **by window**
- Save a run with `np.savez` and reload it

## Tutorial

### 0. RiaB Boilerplate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ratinabox
from ratinabox.Environment import Environment
from ratinabox.Agent import Agent
from ratinabox.Neurons import PlaceCells, HeadDirectionCells

ratinabox.autosave_plots = False
np.random.seed(0)
rng = np.random.default_rng(0)

### 1. Building a maze from walls

An `Environment` is a square boundary plus any internal walls you add. Each wall is a line segment `[[x0, y0], [x1, y1]]`, and `add_wall` appends it. We do not reshape the boundary to make a T-maze. We drop in four walls that seal off the two bottom corners, which leaves a central stem at `x` in `[0.4, 0.6]` and `y` below `0.7`, opening onto a full width arm across the top.

The layout is worth the effort because the geometry is the experimental variable. As one example, subiculum neurons recorded on tracks have been reported to code the current axis of travel (Olson et al., 2017), and that is a question you can only ask in a layout that has an axis.

In [ ]:
env = Environment(params={"scale": 1.0})       # 1 m x 1 m

print("walls on a FRESH env ->", np.asarray(env.walls).shape)

maze_walls = [
    [[0.0, 0.7], [0.4, 0.7]],   # left arm floor
    [[0.6, 0.7], [1.0, 0.7]],   # right arm floor
    [[0.4, 0.0], [0.4, 0.7]],   # left stem wall
    [[0.6, 0.0], [0.6, 0.7]],   # right stem wall
]
for w in maze_walls:
    env.add_wall(w)

print("walls after adding 4 ->", np.asarray(env.walls).shape)
print("we added", len(maze_walls), "and env.walls holds", len(env.walls))

> **Gotcha:** `env.walls` already contains the four **boundary** walls of the square before you add anything. A fresh `Environment` gives `np.asarray(env.walls).shape == (4, 2, 2)`. So `len(env.walls)` is always four more than the number of walls you added, and any code that counts walls has to allow for that. It matters in section 3, where one library check is written in terms of "additional" walls.

In [ ]:
fig, ax = env.plot_environment()
plt.show()

### 2. Keeping the animal in the maze

Walls are hard barriers. The agent's motion model needs no special handling, it simply cannot cross them. Set the starting position inside the stem and run.

Do not trust the picture on its own. Check containment with numbers: every sample recorded below the arm floor should have an `x` inside the stem.

In [ ]:
Ag = Agent(env, params={"dt": 0.05, "speed_mean": 0.1})
Ag.pos = np.array([0.5, 0.35])          # start inside the stem

for _ in range(int(180 / 0.05)):        # 3 minutes at dt = 0.05
    Ag.update()

pos = Ag.get_history_arrays()["pos"]    # (T, 2)
stem = pos[pos[:, 1] < 0.7]             # every sample below the arm floor

print("samples          ->", pos.shape)
print("full y range     -> [%.3f, %.3f]" % (pos[:, 1].min(), pos[:, 1].max()))
print("stem x range     -> [%.3f, %.3f]  (walls sit at 0.4 and 0.6)"
      % (stem[:, 0].min(), stem[:, 0].max()))

sealed = int(((pos[:, 0] < 0.39) & (pos[:, 1] < 0.69)).sum()
             + ((pos[:, 0] > 0.61) & (pos[:, 1] < 0.69)).sum())
print("samples in the sealed corners ->", sealed)

In [ ]:
fig, ax = env.plot_environment()
Ag.plot_trajectory(fig=fig, ax=ax)
plt.show()

### 3. Place fields near walls

In an open arena a Gaussian place field is symmetric in straight line distance. Add a wall and that measure stops describing the animal's world. Two points can be 20 cm apart in a straight line and a long detour apart for the animal. `PlaceCells` takes a `wall_geometry` parameter with three settings.

- `"euclidean"` measures the straight line and ignores walls entirely, so a field bleeds through them.
- `"geodesic"` measures the shortest walk **around** the walls. The field is reduced on the far side but not zeroed, because you can still walk around the end of the wall.
- `"line_of_sight"` is euclidean distance with a hard cutoff. If a wall blocks the straight line the rate is exactly zero.

Here they are on a room with one added wall, with the field centred at `(0.4, 0.3)` and probed at the mirror point `(0.6, 0.3)` straight across the wall.

In [ ]:
env1 = Environment(params={"scale": 1.0})
env1.add_wall([[0.5, 0.0], [0.5, 0.6]])       # ONE added wall, plus the 4 boundary walls
Ag1 = Agent(env1, params={"dt": 0.05})

centre = np.array([[0.4, 0.3]])               # field centre, left of the wall
across = np.array([[0.6, 0.3]])               # probe point, right of the wall

for geom in ["geodesic", "euclidean", "line_of_sight"]:
    pc = PlaceCells(Ag1, params={"n": 1, "widths": 0.3,
                                 "place_cell_centres": centre,
                                 "wall_geometry": geom})
    r = pc.get_state(evaluate_at=None, pos=across).item()
    print(f"{geom:15s} -> {r:.4f}")

Read those three numbers carefully, because they are the whole point of the section. Euclidean puts most of the field's peak straight through a solid wall. Geodesic keeps a little, which is correct, because the animal can walk around the bottom end of this wall and the detour is long but finite. Line of sight cuts it to exactly zero.

> **Gotcha (important):** `geodesic` only works on a room with **one** added wall. On anything bigger RatInABox does not raise, it prints a warning and silently swaps in `line_of_sight`:
>
> ```
> 'geodesic' wall geometry only supported for enivironments with 1 additional wall
> (4 bounding walls + 1 additional). Sorry. Using 'line_of_sight' instead.
> ```
>
> The typo in "enivironments" is really in the library, which is a handy way to be sure you are looking at this message and not one of your own. Ask for `geodesic` on the four wall T-maze and you get `line_of_sight` results under a `geodesic` label. The cell below triggers it on purpose so you know what it looks like.

In [ ]:
Ag_t = Agent(env, params={"dt": 0.05})        # env is the T-maze, 4 added walls
centre_t = np.array([[0.2, 0.4]])             # inside a sealed corner
probe_t = np.array([[0.5, 0.5]])              # inside the stem, across a wall

for geom in ["geodesic", "euclidean", "line_of_sight"]:
    pc = PlaceCells(Ag_t, params={"n": 1, "widths": 0.3,
                                  "place_cell_centres": centre_t,
                                  "wall_geometry": geom})
    r = pc.get_state(evaluate_at=None, pos=probe_t).item()
    print(f"{geom:15s} -> {r:.4f}")

print()
print("geodesic and line_of_sight agree exactly, because geodesic became line_of_sight.")
print("On a multi-wall maze, ask for line_of_sight and mean it.")

### 4. From rates to spikes

Every `Neurons` layer keeps two histories. `firingrate` is the smooth continuous rate in Hz, and `spikes` is the sampled spike train. Build a small population on the maze, drive it, and pull both.

In [ ]:
def tmaze_env():
    """A fresh copy of the T-maze from section 1."""
    e = Environment(params={"scale": 1.0})
    for w in [[[0.0, 0.7], [0.4, 0.7]], [[0.6, 0.7], [1.0, 0.7]],
              [[0.4, 0.0], [0.4, 0.7]], [[0.6, 0.0], [0.6, 0.7]]]:
        e.add_wall(w)
    return e


env4 = tmaze_env()
Ag4 = Agent(env4, params={"dt": 0.05, "speed_mean": 0.1})
Ag4.pos = np.array([0.5, 0.35])

pc4 = PlaceCells(Ag4, params={"n": 5, "widths": 0.15, "max_fr": 10,
                              "wall_geometry": "line_of_sight"})

for _ in range(int(60 / 0.05)):     # 60 s
    Ag4.update()
    pc4.update()

nh = pc4.get_history_arrays()
fr, spk = nh["firingrate"], nh["spikes"]

print("firingrate ->", fr.shape, fr.dtype)
print("spikes     ->", spk.shape, spk.dtype, "  <-- look at that dtype")
print("unique values in spikes ->", np.unique(spk))
print("total spikes ->", int(spk.sum()))

> **Gotcha:** `spikes` is a **boolean** array, not a count. Each entry answers "did this cell fire in this `dt`", so the most you can ever get is one spike per cell per timestep. Do not treat it as a Poisson count. No single entry is ever larger than one, however high the rate goes.

That has a consequence you can measure. Push `max_fr` high enough that a cell would want more than one spike in a single bin and the boolean array cannot hold it, so the recovered rate falls short of the true one.

In [ ]:
dt, n_steps, n_cells = 0.05, 2000, 5

for max_fr in [1, 10, 40]:
    np.random.seed(0)
    e = tmaze_env()
    A = Agent(e, params={"dt": dt, "speed_mean": 0.1})
    A.pos = np.array([0.5, 0.35])
    p = PlaceCells(A, params={"n": n_cells, "widths": 0.15, "max_fr": max_fr,
                              "wall_geometry": "line_of_sight"})
    for _ in range(n_steps):
        A.update()
        p.update()

    h = p.get_history_arrays()
    true_rate = h["firingrate"].mean()                            # Hz
    measured = h["spikes"].sum() / (n_cells * n_steps * dt)       # spikes/cell/s
    print(f"max_fr={max_fr:3d}   mean firingrate {true_rate:6.3f} Hz   "
          f"measured {measured:6.3f} spikes/cell/s   "
          f"shortfall {100 * (1 - measured / true_rate):5.1f} %")

At `max_fr` of 1 and 10 the measured spike rate tracks the mean firing rate closely, and the shortfall is a few percent either side of zero, which is just sampling noise. At 40 it falls short by a visible margin in one direction only. The reason is the boolean bin. With `dt = 0.05` there are only 20 bins per second, so any moment where the cell wants to fire twice in one bin loses a spike. If you need high rates, shrink `dt` or work with `firingrate` directly and skip the sampling.

Here is the population from above as a raster over the smooth rates that produced it.

In [ ]:
t = nh["t"]
window = t <= 20.0                       # first 20 s

fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True,
                        gridspec_kw={"height_ratios": [1, 1.4]})

for c in range(spk.shape[1]):
    st = t[window & spk[:, c]]
    axs[0].scatter(st, np.full_like(st, c), s=6, color="k", marker="|")
axs[0].set_ylabel("cell")
axs[0].set_yticks(range(spk.shape[1]))
axs[0].set_title("spikes (boolean, one row per cell)")

for c in range(fr.shape[1]):
    axs[1].plot(t[window], fr[window, c], lw=0.9, label=f"cell {c}")
axs[1].set_xlabel("time (s)")
axs[1].set_ylabel("rate (Hz)")
axs[1].set_title("the firing rates underneath")
axs[1].legend(ncol=5, fontsize=8)
plt.tight_layout()
plt.show()

Which of the two you save changes the loss a model should use. A firing rate is a smooth continuous target, so squared error is the natural fit. The spike array is boolean, each bin is either zero or one, so a binary likelihood such as binary cross entropy is what matches it. A Poisson negative log likelihood is the standard choice once the bins hold real counts, which is what you get from a finer `dt` or from summing several bins together. Either way squared error on spikes is mis-specified. Save both if you like, but decide which one you are predicting before you write the loss.

### 5. Objects and distance to a goal

Point objects live on the `Environment`. `env.add_object([x, y])` places one and `env.objects["objects"]` reads them back as an `(n_objects, 2)` array. The environment can then measure wall aware distances between two position arrays with `get_distances_between___accounting_for_environment`, which returns a `(len(posA), len(posB))` matrix.

Distance to a goal is one of the more useful derived channels you can build, so it is worth knowing how to get it right on a maze.

In [ ]:
env5 = tmaze_env()
env5.add_object([0.5, 0.95])                          # goal at the top of the stem
goal = np.asarray(env5.objects["objects"]).reshape(1, 2)
print("objects ->", np.asarray(env5.objects["objects"]).shape, goal)

Ag5 = Agent(env5, params={"dt": 0.05, "speed_mean": 0.1})
Ag5.pos = np.array([0.5, 0.35])
for _ in range(int(120 / 0.05)):                      # 2 minutes
    Ag5.update()

h5 = Ag5.get_history_arrays()
traj = h5["pos"]                                      # (T, 2)

d = env5.get_distances_between___accounting_for_environment(
    traj, goal, wall_geometry="line_of_sight")        # (T, 1)

print("distance matrix ->", d.shape)
d = d.reshape(-1)                                     # (T,)
print("range -> [%.3f, %.3f] m" % (d.min(), d.max()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(h5["t"], d, lw=0.8)
ax.set_xlabel("time (s)")
ax.set_ylabel("distance to goal (m)")
ax.set_title("distance to the object at (0.5, 0.95)")
plt.tight_layout()
plt.show()

> **Gotcha:** this method does **not** fall back the way `PlaceCells` does. Ask it for `geodesic` on a maze with more than one added wall and it raises `AssertionError` immediately. The two behaviours are inconsistent, so do not assume that because the place cells accepted `geodesic` the distance call will too.

`line_of_sight` has a quirk of its own. When a wall blocks the straight line it does not return a detour length, it returns the sentinel value `1000`. Nothing is blocked for this particular goal, because it sits above the gap at the top of the stem, which is why the two working geometries agree exactly in the cell below. Move the goal to the end of an arm and the sentinels appear. Problem 4.4 does that and shows how to handle them.

In [ ]:
try:
    env5.get_distances_between___accounting_for_environment(
        traj[:5], goal, wall_geometry="geodesic")
except AssertionError as e:
    print("error class ->", type(e).__name__)
    print("message     ->", str(e)[:110])

print()
print("euclidean and line_of_sight both work on this maze:")
for geom in ["euclidean", "line_of_sight"]:
    dd = env5.get_distances_between___accounting_for_environment(
        traj, goal, wall_geometry=geom).reshape(-1)
    print(f"  {geom:15s} -> [{dd.min():.3f}, {dd.max():.3f}] m, "
          f"{int((dd >= 1000).sum())} blocked samples")

### 6. Packaging a run into arrays

A training set is just aligned arrays. `Ag.get_history_arrays()` gives the trajectory and `layer.get_history_arrays()` gives each population, all on the same time base. Wrap that in one small function so you never write it twice.

In [ ]:
env6 = tmaze_env()
Ag6 = Agent(env6, params={"dt": 0.05, "speed_mean": 0.1})
Ag6.pos = np.array([0.5, 0.35])

# create EVERY layer before stepping anything
hd6 = HeadDirectionCells(Ag6, params={"n": 8, "angular_spread_degrees": 45,
                                      "name": "hd"})
pc6 = PlaceCells(Ag6, params={"n": 6, "widths": 0.15, "max_fr": 10,
                              "wall_geometry": "line_of_sight", "name": "place"})

for _ in range(int(120 / 0.05)):     # 2 minutes, lockstep
    Ag6.update()
    hd6.update()
    pc6.update()


def extract_dataset(Ag, layers):
    """Aligned arrays for one run. Every value shares the same first axis, T."""
    h = Ag.get_history_arrays()
    out = {
        "t": h["t"],                                  # (T,)
        "pos": h["pos"],                              # (T, 2)
        "head_direction": h["head_direction"],        # (T, 2) unit vectors
        "rot_vel": h["rot_vel"],                      # (T,) rad/s
    }
    for layer in layers:
        nh = layer.get_history_arrays()
        out[f"{layer.name}_fr"] = nh["firingrate"]                    # (T, n)
        # spikes come back as bool. The cast is lossless and model code wants numbers.
        out[f"{layer.name}_spikes"] = nh["spikes"].astype(np.float32)  # (T, n)
    return out


ds = extract_dataset(Ag6, [hd6, pc6])

T = ds["t"].shape[0]
for k, v in ds.items():
    print(f"{k:16s} {str(v.shape):12s} {v.dtype}")
    assert v.shape[0] == T, f"{k} does not share T"
print()
print("all", len(ds), "arrays share T =", T)

That `assert` passes only because the agent and both populations were updated **in lockstep**, inside one loop, from the moment the layers existed. Step the agent even once beforehand and its history is one row longer than theirs forever after, which shows up later as a size mismatch from `np.concatenate`. Module 3 walks through that failure in detail.

### 7. Chunking and splitting

Sequence models want windows, not one long run. Slice the `(T, F)` matrix into overlapping windows of length `L` taken every `stride` steps, then split the **windows** into train, validation and test.

In [ ]:
def chunk(arr, L, stride):
    """Overlapping windows of a (T, F) array -> (n_windows, L, F)."""
    starts = range(0, arr.shape[0] - L + 1, stride)
    return np.stack([arr[i:i + L] for i in starts], axis=0)


# inputs: head-direction rates, place rates, and angular velocity as one column
X = np.concatenate([ds["hd_fr"], ds["place_fr"], ds["rot_vel"][:, None]], axis=1)
print("X ->", X.shape)

L, stride = 50, 25
W = chunk(X, L, stride)
print("W ->", W.shape, "= (n_windows, L, F)")

n = W.shape[0]
i1, i2 = int(0.70 * n), int(0.85 * n)
train, val, test = W[:i1], W[i1:i2], W[i2:]

print()
print("train ->", train.shape)
print("val   ->", val.shape)
print("test  ->", test.shape)
print("sum   ->", train.shape[0] + val.shape[0] + test.shape[0], "of", n, "windows")
assert train.shape[0] + val.shape[0] + test.shape[0] == n

Split by **window index**, not by timestep. With `stride < L` the windows overlap, so a split made on the raw timesteps would put the same samples on both sides of the boundary and quietly inflate your validation score. Splitting whole windows keeps each one entirely inside one set. The seam between two adjacent sets still shares `L - stride` timesteps, so drop a window at each boundary if you want the sets fully disjoint.

### 8. Saving

`np.savez` takes keyword arrays and writes one compressed file. Reload it and check.

In [ ]:
from pathlib import Path

here = Path.cwd()
data_dir = here / "data" if (here / "data").is_dir() else here.parent / "data"
data_dir.mkdir(exist_ok=True)
path = data_dir / "module_4_tmaze.npz"

np.savez(path, train=train, val=val, test=test, pos=ds["pos"], t=ds["t"])

z = np.load(path)
print("saved to  ->", path)
print("keys      ->", sorted(z.files))
print("train     ->", z["train"].shape)
assert np.allclose(z["train"], train)
assert np.allclose(z["pos"], ds["pos"])
print("round trip matches")

---

## Key API

| Task | Call |
|---|---|
| Add an internal wall | `env.add_wall([[x0, y0], [x1, y1]])` |
| Read the walls | `env.walls` → `(4 + n_added, 2, 2)`, boundary walls **included** |
| Draw the layout | `env.plot_environment()` → `fig, ax` |
| Overlay a run | `Ag.plot_trajectory(fig=fig, ax=ax)` |
| Start position | `Ag.pos = np.array([x, y])` before the update loop |
| Field geometry | `PlaceCells(Ag, params={"wall_geometry": "line_of_sight"})` |
| Geometry options | `"euclidean"`, `"geodesic"` (**one added wall only**), `"line_of_sight"` |
| Firing at chosen points | `pc.get_state(evaluate_at=None, pos=P)` → `(n, len(P))` |
| Rate map over the env | `pc.get_state(evaluate_at="all")` → `(n, n_pos)` |
| Rates and spikes | `layer.get_history_arrays()` → `t (T,)`, `firingrate (T, n)`, `spikes (T, n)` |
| Spike dtype | `spikes` is **bool**, at most one spike per cell per `dt` |
| Peak rate | `max_fr` in the layer params, in Hz |
| Add an object | `env.add_object([x, y])` |
| Read objects | `env.objects["objects"]` → `(n_objects, 2)` |
| Wall-aware distance | `env.get_distances_between___accounting_for_environment(pA, pB, wall_geometry=...)` |
| Blocked line of sight | that call returns the sentinel `1000`, not a detour length |
| Geodesic distance | raises `AssertionError` on a maze with more than one added wall |
| Trajectory arrays | `Ag.get_history_arrays()` → `t, pos, vel, rot_vel, head_direction, distance_travelled` |
| Layer name | `layer.name`, set with `params={"name": "place"}` |
| Overlapping windows | `chunk(X, L, stride)` → `(n_windows, L, F)` |
| Save and reload | `np.savez(path, train=..., val=...)` then `np.load(path)` |

---

## Problems

Answers are folded below each problem. Try it yourself first.

### Problem 4.1

Build a T-maze from walls, plot it, run an agent inside it for five minutes, and plot the trajectory over the maze.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 4.1"
#| tags: [solution]
np.random.seed(0)

env_41 = Environment(params={"scale": 1.0})
print("fresh env walls ->", np.asarray(env_41.walls).shape)

walls_41 = [
    [[0.0, 0.7], [0.4, 0.7]],   # left arm floor
    [[0.6, 0.7], [1.0, 0.7]],   # right arm floor
    [[0.4, 0.0], [0.4, 0.7]],   # left stem wall
    [[0.6, 0.0], [0.6, 0.7]],   # right stem wall
]
for w in walls_41:
    env_41.add_wall(w)

print("after adding 4  ->", np.asarray(env_41.walls).shape,
      " (4 boundary + 4 added)")
assert np.asarray(env_41.walls).shape == (8, 2, 2)

fig, ax = env_41.plot_environment()
ax.set_title("the T-maze, walls only")
plt.show()

# five minutes at dt = 0.05
Ag_41 = Agent(env_41, params={"dt": 0.05, "speed_mean": 0.1})
Ag_41.pos = np.array([0.5, 0.35])                # start inside the stem
n_steps = int(5 * 60 / 0.05)
for _ in range(n_steps):
    Ag_41.update()

pos_41 = Ag_41.get_history_arrays()["pos"]
stem_41 = pos_41[pos_41[:, 1] < 0.7]
sealed_41 = int(((pos_41[:, 0] < 0.39) & (pos_41[:, 1] < 0.69)).sum()
                + ((pos_41[:, 0] > 0.61) & (pos_41[:, 1] < 0.69)).sum())

fig, ax = env_41.plot_environment()
Ag_41.plot_trajectory(fig=fig, ax=ax)
ax.set_title("5 minutes of movement, constrained by the walls")
plt.show()

print(f"steps                        : {n_steps}")
print(f"samples in the sealed corners: {sealed_41}")
print(f"stem x range while y < 0.7   : [{stem_41[:, 0].min():.3f}, "
      f"{stem_41[:, 0].max():.3f}]   (walls at 0.4 and 0.6)")
print(f"full y range                 : [{pos_41[:, 1].min():.3f}, "
      f"{pos_41[:, 1].max():.3f}]")
assert sealed_41 == 0
print()
print("The animal reaches the full height of the box and the full width of the top")
print("arm, but below y = 0.7 its x never leaves the stem. The walls did all of that")
print("on their own. There is no reward schedule and no special handling in the loop.")

### Problem 4.2

Put a place field near a corner and show it does not leak through the wall. Contrast the three `wall_geometry` settings and report the firing rate on the far side of the wall.

**Hint:** do the true geodesic comparison on a room with a single added wall, because on a multi-wall maze `geodesic` silently becomes `line_of_sight`. Evaluate at a specific point with `get_state(evaluate_at=None, pos=...)`.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 4.2"
#| tags: [solution]
np.random.seed(0)

# ---- part 1: the honest three-way contrast, on a room with ONE added wall ----
env_42a = Environment(params={"scale": 1.0})
env_42a.add_wall([[0.5, 0.0], [0.5, 0.6]])
Ag_42a = Agent(env_42a, params={"dt": 0.05})

centre_a = np.array([[0.4, 0.3]])       # field centre, left of the wall
across_a = np.array([[0.6, 0.3]])       # mirror point, right of the wall

rates_a = {}
for geom in ["euclidean", "geodesic", "line_of_sight"]:
    pc = PlaceCells(Ag_42a, params={"n": 1, "widths": 0.3,
                                    "place_cell_centres": centre_a,
                                    "wall_geometry": geom})
    rates_a[geom] = pc.get_state(evaluate_at=None, pos=across_a).item()
    if geom == "geodesic":
        pc_geo_a = pc

print("ONE added wall, field at (0.4, 0.3), probed at (0.6, 0.3):")
for geom, r in rates_a.items():
    print(f"  {geom:15s} -> {r:.4f}")
assert rates_a["euclidean"] > rates_a["geodesic"] > rates_a["line_of_sight"]

pc_geo_a.plot_rate_map(chosen_neurons="1")
plt.suptitle("geodesic field, one added wall", y=1.02)
plt.show()

# ---- part 2: a corner of the real T-maze, where geodesic is not available ----
env_42b = Environment(params={"scale": 1.0})
for w in [[[0.0, 0.7], [0.4, 0.7]], [[0.6, 0.7], [1.0, 0.7]],
          [[0.4, 0.0], [0.4, 0.7]], [[0.6, 0.0], [0.6, 0.7]]]:
    env_42b.add_wall(w)
Ag_42b = Agent(env_42b, params={"dt": 0.05})

corner = np.array([[0.25, 0.80]])       # in the left arm, just above the arm floor
behind = np.array([[0.25, 0.60]])       # sealed corner, straight through that wall

rates_b = {}
for geom in ["euclidean", "geodesic", "line_of_sight"]:
    pc = PlaceCells(Ag_42b, params={"n": 1, "widths": 0.25,
                                    "place_cell_centres": corner,
                                    "wall_geometry": geom})
    rates_b[geom] = pc.get_state(evaluate_at=None, pos=behind).item()
    if geom == "line_of_sight":
        pc_los_b = pc

print()
print("T-maze (4 added walls), field at (0.25, 0.80), probed at (0.25, 0.60):")
for geom, r in rates_b.items():
    print(f"  {geom:15s} -> {r:.4f}")

pc_los_b.plot_rate_map(chosen_neurons="1")
plt.suptitle("line_of_sight field near the left arm corner", y=1.02)
plt.show()

print()
print("With one added wall all three settings differ, and they differ in the way the")
print("definitions predict. Euclidean puts most of the peak straight through solid")
print("wall. Geodesic keeps a small amount, which is right, because the animal can")
print("walk around the free end of the wall and the detour is long but finite. Line")
print("of sight cuts it to exactly zero.")
print()
print("On the T-maze the geodesic and line_of_sight numbers are identical, and the")
print("warning printed above says why: geodesic is only supported with one additional")
print("wall, so RatInABox substituted line_of_sight without raising. That is why the")
print("multi-wall case should ask for line_of_sight directly. Either way the field")
print("does not leak into the sealed corner, while euclidean happily does.")

### Problem 4.3

Generate rates and spikes for a place population and plot a spike raster above the underlying rates for five cells over twenty seconds.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 4.3"
#| tags: [solution]
np.random.seed(0)

env_43 = Environment(params={"scale": 1.0})
for w in [[[0.0, 0.7], [0.4, 0.7]], [[0.6, 0.7], [1.0, 0.7]],
          [[0.4, 0.0], [0.4, 0.7]], [[0.6, 0.0], [0.6, 0.7]]]:
    env_43.add_wall(w)

Ag_43 = Agent(env_43, params={"dt": 0.05, "speed_mean": 0.15})
Ag_43.pos = np.array([0.5, 0.30])

# tile 5 fields up the stem so all of them are visited inside a short run
centres_43 = np.array([[0.5, 0.08], [0.5, 0.20], [0.5, 0.32],
                       [0.5, 0.44], [0.5, 0.56]])
pc_43 = PlaceCells(Ag_43, params={"n": 5, "widths": 0.12, "max_fr": 10,
                                  "wall_geometry": "line_of_sight",
                                  "place_cell_centres": centres_43})

n_43 = int(20 / 0.05)                    # 20 s at dt = 0.05
for _ in range(n_43):
    Ag_43.update()
    pc_43.update()

h_43 = pc_43.get_history_arrays()
t_43, fr_43, spk_43 = h_43["t"], h_43["firingrate"], h_43["spikes"]

assert fr_43.shape == (n_43, 5)
assert spk_43.shape == (n_43, 5)
assert spk_43.dtype == bool
assert spk_43.sum() > 0

fig, axs = plt.subplots(2, 1, figsize=(9, 5.2), sharex=True,
                        gridspec_kw={"height_ratios": [1, 1.5]})
for c in range(5):
    st = t_43[spk_43[:, c]]
    axs[0].scatter(st, np.full_like(st, c), s=40, color="k", marker="|")
axs[0].set_ylabel("cell")
axs[0].set_yticks(range(5))
axs[0].set_ylim(-0.6, 4.6)
axs[0].set_title("spike raster, 5 place cells over 20 s")

for c in range(5):
    axs[1].plot(t_43, fr_43[:, c], lw=1.0, label=f"cell {c}")
axs[1].set_xlabel("time (s)")
axs[1].set_ylabel("firing rate (Hz)")
axs[1].set_title("the rates that generated them")
axs[1].legend(ncol=5, fontsize=8)
plt.tight_layout()
plt.show()

print(f"firingrate  : {fr_43.shape}  {fr_43.dtype}")
print(f"spikes      : {spk_43.shape}  {spk_43.dtype}   <-- boolean, not a count")
print(f"total spikes: {int(spk_43.sum())} over 20 s across 5 cells")
print(f"per cell    : {spk_43.sum(axis=0).astype(int).tolist()}")
print(f"peak rate   : {fr_43.max():.2f} Hz against max_fr = 10")
print()
print("Each cell's raster is densest exactly where its rate curve peaks, which is what")
print("you want to see. Cells the animal visits less often collect fewer spikes. Since")
print("spikes is boolean, a row can hold at most one spike per cell per dt, so at")
print("dt = 0.05 no cell can show more than 20 spikes per second no matter how high")
print("max_fr goes.")

### Problem 4.4

Add an object at the end of one arm and plot distance to it over time.

**Hint:** use `wall_geometry="line_of_sight"`. Geodesic raises `AssertionError` once the maze has more than one added wall.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 4.4"
#| tags: [solution]
np.random.seed(0)

env_44 = Environment(params={"scale": 1.0})
for w in [[[0.0, 0.7], [0.4, 0.7]], [[0.6, 0.7], [1.0, 0.7]],
          [[0.4, 0.0], [0.4, 0.7]], [[0.6, 0.0], [0.6, 0.7]]]:
    env_44.add_wall(w)

env_44.add_object([0.9, 0.85])                            # end of the right arm
goal_44 = np.asarray(env_44.objects["objects"]).reshape(1, 2)
print("objects ->", np.asarray(env_44.objects["objects"]).shape, goal_44)

Ag_44 = Agent(env_44, params={"dt": 0.05, "speed_mean": 0.1})
Ag_44.pos = np.array([0.5, 0.35])
for _ in range(int(120 / 0.05)):                          # 2 minutes
    Ag_44.update()

h_44 = Ag_44.get_history_arrays()
traj_44, t_44 = h_44["pos"], h_44["t"]

d_los = env_44.get_distances_between___accounting_for_environment(
    traj_44, goal_44, wall_geometry="line_of_sight").reshape(-1)
d_euc = env_44.get_distances_between___accounting_for_environment(
    traj_44, goal_44, wall_geometry="euclidean").reshape(-1)

# geodesic is a HARD failure here, not a fallback
try:
    env_44.get_distances_between___accounting_for_environment(
        traj_44[:5], goal_44, wall_geometry="geodesic")
except AssertionError as e:
    print("geodesic ->", type(e).__name__, ":", str(e)[:90])

# line_of_sight returns the sentinel 1000 whenever a wall blocks the view
blocked = d_los >= 1000
d_plot = np.where(blocked, np.nan, d_los)

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(t_44, d_euc, lw=0.8, color="C1", label="euclidean (ignores walls)")
ax.plot(t_44, d_plot, lw=1.2, color="C0", label="line_of_sight (visible only)")
ax.fill_between(t_44, 0, 1, where=blocked, transform=ax.get_xaxis_transform(),
                color="0.85", zorder=0, label="goal out of sight")
ax.set_xlabel("time (s)")
ax.set_ylabel("distance to goal (m)")
ax.set_title("distance to an object at the end of the right arm")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print()
print(f"samples            : {d_los.shape[0]}")
print(f"euclidean range    : [{d_euc.min():.3f}, {d_euc.max():.3f}] m")
print(f"blocked samples    : {int(blocked.sum())} of {blocked.size} "
      f"({100 * blocked.mean():.1f} %)")
print(f"visible-only range : [{np.nanmin(d_plot):.3f}, {np.nanmax(d_plot):.3f}] m")
print()
print("Two things to take away. Geodesic is not an option on this maze at all, it")
print("raises AssertionError rather than falling back the way PlaceCells does. And")
print("line_of_sight does not return a detour length when a wall is in the way, it")
print("returns the sentinel 1000, which is the shaded region above. Every moment the")
print("animal is low in the stem with the right arm floor between it and the goal is")
print("a sentinel, and here that is about a fifth of the run. Mask them before you")
print("plot or average, otherwise a single 1000 will wreck any statistic you compute.")
print("If you want a distance channel with no gaps on a multi-wall maze, euclidean is")
print("the honest choice, as long as you say that it ignores walls.")

### Problem 4.5

Write `extract_dataset(Ag, layers)` returning aligned arrays, chunk the result into overlapping windows, and produce train, validation and test splits.

**Hint:** update the agent and every layer in one lockstep loop or the histories will differ in length. Split by window index rather than by timestep.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 4.5"
#| tags: [solution]
np.random.seed(0)

# ---- a run on the T-maze with two populations ----
env_45 = Environment(params={"scale": 1.0})
for w in [[[0.0, 0.7], [0.4, 0.7]], [[0.6, 0.7], [1.0, 0.7]],
          [[0.4, 0.0], [0.4, 0.7]], [[0.6, 0.0], [0.6, 0.7]]]:
    env_45.add_wall(w)

Ag_45 = Agent(env_45, params={"dt": 0.05, "speed_mean": 0.1})
Ag_45.pos = np.array([0.5, 0.35])

# every layer is created BEFORE anything is stepped
hd_45 = HeadDirectionCells(Ag_45, params={"n": 8, "angular_spread_degrees": 45,
                                          "name": "hd"})
pc_45 = PlaceCells(Ag_45, params={"n": 6, "widths": 0.15, "max_fr": 10,
                                  "wall_geometry": "line_of_sight",
                                  "name": "place"})

T_45 = int(180 / 0.05)                  # 3 minutes
for _ in range(T_45):                   # lockstep: one loop, everything updated
    Ag_45.update()
    hd_45.update()
    pc_45.update()


# ---- the extractor ----
def extract_dataset(Ag, layers):
    """Aligned arrays for one run. Every value shares the same first axis, T."""
    h = Ag.get_history_arrays()
    out = {
        "t": h["t"],                                   # (T,)
        "pos": h["pos"],                               # (T, 2)
        "head_direction": h["head_direction"],         # (T, 2) unit vectors
        "rot_vel": h["rot_vel"],                       # (T,) rad/s
    }
    for layer in layers:
        nh = layer.get_history_arrays()
        out[f"{layer.name}_fr"] = nh["firingrate"]                     # (T, n)
        out[f"{layer.name}_spikes"] = nh["spikes"].astype(np.float32)  # (T, n)
    return out


ds_45 = extract_dataset(Ag_45, [hd_45, pc_45])

T = ds_45["t"].shape[0]
print("extract_dataset output")
for k, v in ds_45.items():
    print(f"  {k:16s} {str(v.shape):12s} {v.dtype}")
    assert v.shape[0] == T, f"{k} does not share T"
print(f"  all {len(ds_45)} arrays share T = {T}, and T == steps run: {T == T_45}")


# ---- chunk into overlapping windows ----
def chunk(arr, L, stride):
    """Overlapping windows of a (T, F) array -> (n_windows, L, F)."""
    starts = range(0, arr.shape[0] - L + 1, stride)
    return np.stack([arr[i:i + L] for i in starts], axis=0)


X_45 = np.concatenate(
    [ds_45["hd_fr"], ds_45["place_fr"], ds_45["rot_vel"][:, None]], axis=1)
F = X_45.shape[1]

L, stride = 50, 25
W_45 = chunk(X_45, L, stride)
n_win = W_45.shape[0]

print()
print(f"X            : {X_45.shape}   = (T, {hd_45.n} hd + {pc_45.n} place + 1 rot_vel)")
print(f"windows      : {W_45.shape}   = (n_windows, L, F) with L={L}, stride={stride}")
assert W_45.shape == (n_win, L, F)
assert np.allclose(W_45[0], X_45[:L])
assert np.allclose(W_45[1], X_45[stride:stride + L])


# ---- split BY WINDOW ----
i1, i2 = int(0.70 * n_win), int(0.85 * n_win)
train_45, val_45, test_45 = W_45[:i1], W_45[i1:i2], W_45[i2:]

print()
print(f"train        : {train_45.shape}")
print(f"val          : {val_45.shape}")
print(f"test         : {test_45.shape}")
total = train_45.shape[0] + val_45.shape[0] + test_45.shape[0]
print(f"sum of splits: {total}  against n_windows = {n_win}   match: {total == n_win}")
assert total == n_win
assert train_45.shape[1:] == val_45.shape[1:] == test_45.shape[1:] == (L, F)

# save and reload, so the whole thing is one reproducible artefact
from pathlib import Path

here_45 = Path.cwd()
dd_45 = here_45 / "data" if (here_45 / "data").is_dir() else here_45.parent / "data"
dd_45.mkdir(exist_ok=True)
p_45 = dd_45 / "module_4_problem_45.npz"
np.savez(p_45, train=train_45, val=val_45, test=test_45,
         pos=ds_45["pos"], t=ds_45["t"])
z_45 = np.load(p_45)
assert np.allclose(z_45["train"], train_45)
print(f"saved and reloaded {p_45.name}, train matches: True")

print()
print("Two rules make this work. The agent and both populations were updated in one")
print("lockstep loop, so every history has exactly T rows and the assert on shared T")
print("passes without any resampling. And the split is on window index, not timestep.")
print(f"With stride {stride} below L {L} the windows overlap by {L - stride} steps, so a")
print("split made on raw timesteps would put the same samples in train and in val and")
print("inflate the validation score. Splitting whole windows keeps each window inside")
print("one set. The two seams still share a few timesteps, so drop one window at each")
print("boundary if you need the sets completely disjoint.")

# References

@article{george2024ratinabox,
  author  = {George, Tom M. and Rastogi, Mehul and de Cothi, William and
             Clopath, Claudia and Stachenfeld, Kimberly and Barry, Caswell},
  title   = {{RatInABox}, a toolkit for modelling locomotion and neuronal
             activity in continuous environments},
  journal = {eLife},
  volume  = {13},
  pages   = {e85274},
  year    = {2024},
  doi     = {10.7554/eLife.85274}
}

@article{wood2000hippocampal,
  author  = {Wood, Emma R. and Dudchenko, Paul A. and Robitsek, R. Jonathan and
             Eichenbaum, Howard},
  title   = {Hippocampal neurons encode information about different types of
             memory episodes occurring in the same location},
  journal = {Neuron},
  volume  = {27},
  number  = {3},
  pages   = {623--633},
  year    = {2000},
  doi     = {10.1016/S0896-6273(00)00071-4}
}

@article{frank2000trajectory,
  author  = {Frank, Loren M. and Brown, Emery N. and Wilson, Matthew},
  title   = {Trajectory encoding in the hippocampus and entorhinal cortex},
  journal = {Neuron},
  volume  = {27},
  number  = {1},
  pages   = {169--178},
  year    = {2000},
  doi     = {10.1016/S0896-6273(00)00018-0}
}

@article{mcnaughton1983contributions,
  author  = {McNaughton, B. L. and Barnes, C. A. and O'Keefe, J.},
  title   = {The contributions of position, direction, and velocity to single
             unit activity in the hippocampus of freely-moving rats},
  journal = {Experimental Brain Research},
  volume  = {52},
  number  = {1},
  pages   = {41--49},
  year    = {1983},
  doi     = {10.1007/BF00237147}
}

@article{olson2017subiculum,
  author  = {Olson, Jake M. and Tongprasearth, Kanyanat and Nitz, Douglas A.},
  title   = {Subiculum neurons map the current axis of travel},
  journal = {Nature Neuroscience},
  volume  = {20},
  number  = {2},
  pages   = {170--172},
  year    = {2017},
  doi     = {10.1038/nn.4464}
}